# Build 03-02 · Reweight mitigation — the corrector, per axis, on the exported training split

Runs in the **analysis `.venv`** (`python3` kernel), loads no model. Reads the
**treatment-joined training material** built by 03_01 (`corrector_targets` kind), runs the four
schemes, and writes one corrected target per axis into `src/data/real/mitigation/`. Retraining is
**03_03_retrain.ipynb**, on each version's own kernel.

```
inputs/features_<v>_<split>.parquet   +   mitigation/inputs/corrector_targets_<v>_<split>.parquet
   ──▶  §3 ReweightCorrector per axis (analysis env)
   ──▶  mitigation/<v>_corrected_<split>_<tag>.parquet  (+ _meta.json)
   ──▶  03_03_retrain.ipynb  (env-v2 / env-v3 kernels, separately)
```

The axes (full parameter definition in `src/mitigator/corrector/reweight.py`):

| scheme | garage rows | model-scrapped rows (U) |
|---|---|---|
| naive | (observed, 1) | (1, 1) — the contaminated baseline |
| rarity | (observed, m_c) | (1, m_c) — Axis A only, labels kept |
| transport | (observed, 1) | (1, g) + (0, 1−g) — Axis B soft split |
| pnu | (observed, m_c) | (1, g·m_c) + (0, (1−g)·m_c) — combined |

U is the **recorded `decision`**, never τ — naive/transport are identical under both τ modes and
run once; only rarity/pnu run twice (`regime` = per-row `config.threshold_on(DECIDER, date)`,
`fixed` = one scalar, `TAU_FIXED` — v2's own 0.872 by default, see §1).

**Per-version feasibility** (thesis `tab:scheme-feasibility`): **v3** has score + decision (v2
serving log) → all four schemes. **v2** has decision only (vehicle-status file; the v1-era
deciding score is destroyed) → **naive/transport only** — §3 skips rarity/pnu automatically when
the input carries no `score`. v1 is out of scope (pre-model labels; no retraining).

**Why `naive` runs for BOTH v2 and v3, not just as a formality.** `naive` here is NOT a stand-in
for Allianz's actual historical production model — it is the **uncorrected control arm on the
exact same `corrector_targets` population every other scheme in this run uses**. §1's
`TARGETS_PATH` is one file, read once, and every entry in `RUNS` (naive included) is fit on it.
That population is already smaller than each version's real historical training set: 03_01 §3
drops v3 claim_ids the v2 serving log never covers, and 03_01 §4 drops v2's
awaiting-authorisation/unrecovered/NaN statuses before `naive` ever sees the data — so v2's
`naive` is "the v2 methodology refit on known-outcome rows only" and v3's `naive` is "the v3
methodology refit on v2-log-matched rows only", neither identical to the real deployed model.
Comparing `rarity`/`transport`/`pnu` against Allianz's actual original model would confound "the
correction worked" with "the training population changed" (dropped rows are not random — see
03_01's `missing_claim_ids`/`observed_mismatch` diagnostics). Running `naive` on the SAME
`corrector_targets` file isolates the correction's own effect: every downstream comparison in
03_04/03_05 is naive-vs-corrected on one identical population, per version.

### 실행 전 설정

**커널**: analysis `.venv` (`python3`)

**바꿔야 할 것 (§1)**:
- `VERSION` — `"v2"` 또는 `"v3"` (v2는 score가 없어서 naive/transport 두 axis만 자동으로 실행됨)
- `SPLIT` — 그대로 `"train"` 두세요 — 03_03이 재훈련할 split과 반드시 같아야 합니다.
- `DECIDER`(기본 `"v2"`)는 보통 안 건드려도 됩니다.

**선행 조건**: `03_01_corrector_inputs.ipynb`가 이 `VERSION`+`SPLIT`에 대한 `corrector_targets`를
이미 만들어놨어야 합니다 (안 그러면 "run 03_01_corrector_inputs first" 에러).

In [ ]:
# §0 — setup (analysis .venv kernel)
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import threshold
from mitigator.corrector.reweight import ReweightCorrector
import figstyle

pd.set_option("display.width", 160)
print("ROOT =", ROOT)

In [ ]:
# §1 — RUN_SPEC: every knob of this run lives HERE. Paths resolve through config.
VERSION = "v3"        # "v2" runs naive/transport only (no deciding score — see header)
SPLIT = "train"       # one of config.SPLITS[VERSION] — the split 03_03 will retrain on
DECIDER = "v2"        # whose regime record resolves tau_i for the rarity band (v3's labels)
ID_COL = "claim_id"

FEATURES_PATH = config.split_path("processed_inputs", VERSION, SPLIT)
TARGETS_PATH = config.split_path("corrector_targets", VERSION, SPLIT)   # built by 03_01

# the axes: (scheme, tau_mode). naive/transport ignore tau -> run once.
RUNS = [("naive", None), ("rarity", "regime"), ("rarity", "fixed"),
        ("transport", None), ("pnu", "regime"), ("pnu", "fixed")]
BAND_H, CLIP_LO, CLIP_HI = 0.01, 0.25, 4.0   # fallbacks — §2c overrides when SELECT_PARAMS
# fixed mode: one scalar tau for every row, regardless of when it happened. 0.872 (2026-09-13,
# user-set) is v2's OWN most-cited single threshold -- it is the value TWO of v2's five regimes
# actually ran (2021-06-03 -> 2024-06-02, then again 2026-02-25 -> 2026-06-30 --
# config.DECISION_RULES["v2"]["regimes"]), so "pretend it was always one number" reads most
# naturally as "pretend it was always THIS number", not an empirically dredged pooled statistic.
# None still falls back to threshold.read_off() (min score among decision=1, pooled across
# whatever eras this split spans) -- that logic is untouched, just no longer the default here.
TAU_FIXED = 0.872

# §2c selection knobs (policy in §1b): power + variance criteria ONLY — never metrics computed
# on this split's (contaminated) labels. Axis A only: band_h/clip touch just rarity & pnu.
SELECT_PARAMS = True                                        # False -> run on the fallbacks above
BAND_H_GRID = [0.005, 0.0075, 0.01, 0.015, 0.02, 0.03, 0.05, 0.075, 0.1]
CLIP_HI_GRID = [1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0, 20.0]
MIN_CELL_N = 100      # power gate: fewest verified rows cell 1 may rest on (RSE ≈ 1/√n: 10% at
                      # 100 vs 14% at 50; and the v3 train window spans TWO decider regimes, so
                      # a pooled 50 could leave ~25 rows per era — rationale in §1b)
H_MAX = 0.05          # locality cap: §2c never widens the band past this to satisfy MIN_CELL_N
MIN_ESS_FRAC = 0.50   # variance gate: floor on ESS(w)/n over the whole split


def run_tag(scheme: str, mode: str | None) -> str:
    """Filename tag for one axis run, e.g. 'transport', 'rarity_regime'."""
    return scheme if mode is None else scheme + "_" + mode


def corrected_path(tag: str) -> Path:
    """config's corrected kind + split, with the axis tag before the extension."""
    base = config.split_path("corrected", VERSION, SPLIT)
    return base.with_name(base.stem + "_" + tag + base.suffix)


print("features :", FEATURES_PATH)
print("targets  :", TARGETS_PATH)
print("out      :", corrected_path("<tag>"))
figstyle.apply()
figstyle.FIG_DIR = ROOT / "figures" / "mitigation" / "03_02" / VERSION
# ALIAS_SPLIT is a figstyle module global -- if this kernel previously ran 00_SHAP.ipynb or a
# 04_0x SHAP-DiD notebook without a restart, it would still be True here and band_table/
# clip_table/diag_summary (none of which name a feature) would wrongly nest under
# FIG_DIR/real_named/ instead of saving flat, as this notebook's own §4 documents.
figstyle.ALIAS_SPLIT = False
print("figures  :", figstyle.FIG_DIR)


## §1b — choosing BAND_H / CLIP_LO / CLIP_HI

"Best" here can never mean best downstream metric: every metric computable on this split is
measured against SFP-contaminated labels, so tuning h/clip on them would fit the corrector to
the very labels it distrusts — the same reason τ is never tuned on IPS-corrected data.
Selection therefore uses only **pre-correction power and variance diagnostics**, and 03_04
keeps h/clip as sensitivity axes, not tuned constants.

- **BAND_H** — the band exists so cell 1 (verified repairable just under τ) holds enough rows
  to estimate its frequency. Rule: the **smallest** h in `BAND_H_GRID` with
  n(cell 1) ≥ `MIN_CELL_N`, **restricted to h ≤ `H_MAX`** — any wider only dilutes "boundary"
  with ordinary mid-score rows.
  - *Why 100, not 50:* the cell-frequency RSE is ≈ 1/√n — 14% at 50, 10% at 100 — and the v3
    train window spans **two** τ regimes of the v2 decider (0.872 until 2024-06-02, then
    0.825), so a pooled 50 could leave ~25 rows per era; 100 keeps the per-era cell 1
    reportable beside §2b's era audit. The gate is a **credibility floor, not a precision
    knob**: at these frequencies raw m₁ = 1/(K·f₁) exceeds any admissible clip_hi by orders of
    magnitude, so the clipped weight is identical whether cell 1 holds 50 or 500 rows.
  - *Why cap h:* widening the band is the *only* way to satisfy the gate, so without a cap the
    gate could always be met by de-localising the band. If no h ≤ `H_MAX` reaches
    `MIN_CELL_N`, that is a **finding** — the verified edge really is thin, the
    label-starvation face of the SFP — not a parameter problem: keep the fallback h and carry
    rarity/pnu as low-power (§4 caveat).
- **CLIP_LO is a non-choice.** m_c = 1/(K·f_c) with f_c ≤ 1 and K ≤ 4 non-empty cells, so every
  multiplier is ≥ 1/K ≥ 0.25 *before* clipping: a floor at 0.25 can never bind. It stays 0.25.
- **CLIP_HI** — unclipped, every cell carries exactly n/K of the total weight
  (n_c·m_c = n_c·n/(K·n_c) = n/K): that equal-cell-mass balance IS the design intent. Clipping
  pulls the rare cells back toward naive to control variance. Rule: the **largest** hi in
  `CLIP_HI_GRID` with ESS(w)/n ≥ `MIN_ESS_FRAC` — the most faithful reweighting the variance
  budget allows (ESS defined below).
- The **fixed** τ arm inherits the regime arm's choice: selection runs once on the faithful
  regime τ, so the fixed-vs-regime contrast downstream stays a *pure* τ sensitivity.

**Vocabulary for the §2c tables** — *weight* wᵢ is the per-row multiplier (the parquet's
`weight` column); *mass* is the summed weight of a row set, Σᵢ wᵢ over that set. A cell's mass
is n_c·m_c, and unclipped every cell's mass is exactly n/K (the n_c cancels). Columns:
`n_cell*` = rows per cell at each candidate h (the power gate reads `n_cell1`); `mass_cell*` =
each cell's share of total mass at each candidate ceiling (unclipped ⇒ 1/K each; clipping pulls
the rare cells' share back down); `max_mult` = largest per-row weight; `hi_binds` = whether the
ceiling actually cuts anything; `w_pos_share` = weighted share of label 1 (what the retrain
sees); `ess_frac` = ESS(w)/n.

**ESS (Kish 1992, bib key s4).** ESS(w) = (Σᵢ wᵢ)² / Σᵢ wᵢ² — the number of *equally*-weighted
rows whose estimates would fluctuate as much, since Var(weighted mean) ∝ Σwᵢ²/(Σwᵢ)² = 1/ESS.
Equal weights give ESS = n; a few huge weights collapse it toward the count of those rows (they
enter Σw² with *squared* multipliers). n/ESS is the *design effect*. `MIN_ESS_FRAC` = 0.5
therefore reads: **the reweighting may at most double the variance** (design effect ≤ 2) — a
declared budget, not a theorem: if the chosen ceiling flips when 0.5 moves to 0.4/0.6, report
that sensitivity. Thesis: concept in §2.6 (bg-causal), usage in §3.5.2 (subsec:ips).

The helpers below are pure functions; the sweep itself is **§2c** (it needs §2's load) and
overrides §1's fallbacks in place before §3 runs. The values actually used land in every
run's `_meta.json`, so 03_03/03_04 read them from the sidecar, never from this notebook.

In [ ]:
# §1b — selection helpers (pure; §2c applies them to the loaded split)
def ess_frac(w: np.ndarray) -> float:
    """Effective sample size (Σw)²/Σw² of a weight vector, as a fraction of len(w)."""
    return float(w.sum() ** 2 / (w * w).sum() / len(w))


def band_table(score: np.ndarray, tau: np.ndarray, y: np.ndarray,
               grid: list[float]) -> pd.DataFrame:
    """Cell occupancy per candidate h — the cell rule mirrors ReweightCorrector.correct."""
    rows = []
    for h in sorted(grid):
        cell = np.where(score >= tau - h, 1 + y, 3 + y)
        n = pd.Series(cell).value_counts()
        rows.append({"band_h": h, **{f"n_cell{c}": int(n.get(c, 0)) for c in (1, 2, 3, 4)},
                     "cell1_pass": bool(n.get(1, 0) >= MIN_CELL_N)})
    return pd.DataFrame(rows).set_index("band_h")


def clip_table(cell: np.ndarray, grid_hi: list[float], lo: float, raw_max: float) -> pd.DataFrame:
    """ESS / max multiplier / per-cell weight mass per candidate clip_hi.

    Multipliers come from the corrector's own _rarity, so the sweep cannot drift from §3.
    Unclipped every cell holds 1/K of the mass; mass_cell* shows what clipping trades away.
    """
    rows = []
    for hi in sorted(grid_hi):
        mult = ReweightCorrector(scheme="rarity", clip_lo=lo, clip_hi=hi)._rarity(cell)
        w = pd.Series(cell).map(mult).to_numpy(dtype=float)
        rows.append({"clip_hi": hi, "hi_binds": bool(raw_max > hi),
                     "ess_frac": round(ess_frac(w), 3),
                     "max_mult": round(max(mult.values()), 3),
                     "w_pos_share": round(float(w[(cell == 2) | (cell == 4)].sum() / w.sum()), 4),
                     **{f"mass_cell{int(c)}": round(float(w[cell == c].sum() / w.sum()), 3)
                        for c in sorted(mult)},
                     "ess_pass": bool(ess_frac(w) >= MIN_ESS_FRAC)})
    return pd.DataFrame(rows).set_index("clip_hi")

## §2 — load & audit

`corrector_targets` arrives with the treatment already joined (03_01 dropped the rows its source
never covered — counts in that file's `_meta.json`). Here: canonical checks, P/N/U, and — when a
score column exists — the per-era boundary audit (`threshold.read_off` **per regime**, never
pooled) and the band-occupancy power gate for the rarity scheme's cell 1.

In [ ]:
# §2a — load + canonical checks
features = pd.read_parquet(FEATURES_PATH)
assert TARGETS_PATH.is_file(), f"{TARGETS_PATH} missing — run 03_01_corrector_inputs first"
corr_targets = pd.read_parquet(TARGETS_PATH)

need = [ID_COL, "date", "observed", "decision"]
miss = [c for c in need if c not in corr_targets.columns]
assert not miss, f"corrector_targets is missing {miss} — rebuild it with 03_01"
assert ID_COL in features.columns
HAS_SCORE = "score" in corr_targets.columns

corr_targets = corr_targets.copy()
corr_targets["date"] = pd.to_datetime(corr_targets["date"])
if getattr(corr_targets["date"].dt, "tz", None) is not None:
    # drop the zone, keep the wall clock — same rule as threshold.apply / ReweightCorrector._tau,
    # so the era bins and the tau LUT below stay comparable with tz-naive regime dates
    corr_targets["date"] = corr_targets["date"].dt.tz_localize(None)
d = corr_targets["date"]
print(f"{VERSION} {SPLIT}: {len(corr_targets):,} rows, {d.min():%Y-%m-%d} -> {d.max():%Y-%m-%d}"
      f" | score column: {HAS_SCORE}")
if not HAS_SCORE:
    print("no deciding score -> rarity/pnu will be SKIPPED (v2 reality; thesis tab:scheme-feasibility)")

In [ ]:
# §2b — P/N/U, and (score only) per-era boundary audit + band occupancy
dec_col = corr_targets["decision"].astype(int)
obs_col = corr_targets["observed"].astype(int)
P = int(((dec_col == 0) & (obs_col == 1)).sum())
N = int(((dec_col == 0) & (obs_col == 0)).sum())
U = int((dec_col == 1).sum())
bad = int(((dec_col == 1) & (obs_col == 0)).sum())
warn = f"  !! scrapped-but-observed=0: {bad}" if bad else ""
print(f"P={P:,}  N={N:,}  U={U:,} ({U / len(corr_targets):.2%}){warn}")

if HAS_SCORE:
    days = d.dt.normalize()
    # pd.Timestamp(day): Series.unique() hands back numpy datetime64 for a tz-NAIVE column on
    # some pandas versions (a DatetimeArray of Timestamps when tz-aware), and only Timestamp
    # has .date()
    tau_lut = {day: config.threshold_on(DECIDER, str(pd.Timestamp(day).date()))
               for day in days.unique()}
    tau_row = days.map(tau_lut).astype(float)

    # era index = how many declared breaks this row's day is at or past. Plain >= comparisons
    # against the regime dates — no bin edges, so nothing has to reconcile the column's unit
    # (the export is datetime64[us]) with ns-based Timestamp.min/max.
    brks = [pd.Timestamp(b["date"]) for b in config.breaks(DECIDER)]
    era = pd.Series(0, index=d.index)
    for brk in brks:
        era += (days >= brk).astype(int)

    def era_label(k: int) -> str:
        """Half-open [from, until) of era k, matching threshold.apply()'s regime semantics."""
        lo = brks[k - 1].date() if k else "-inf"
        hi = brks[k].date() if k < len(brks) else "+inf"
        return f"[{lo}, {hi})"

    rows = []
    for e, g in corr_targets.groupby(era):
        gd = g["decision"].astype(int)
        r = {"era": era_label(int(e)), "tau_declared": float(tau_row[g.index].iloc[0]),
             "n": len(g), "n_scrapped": int(gd.sum())}
        if 0 < gd.sum() < len(g):
            ro = threshold.read_off(g)
            r |= {"applied_tau": ro["tau"], "deterministic": ro["deterministic"],
                  "overlap_rows": ro["overlap_rows"]}
        rows.append(r)
    display(pd.DataFrame(rows))
    for brk in config.spans_a_break(DECIDER, str(d.min().date()), str(d.max().date())):
        print("regime break inside this split:", brk)

    in_band = (dec_col == 0) & (corr_targets["score"] >= tau_row - BAND_H)
    print(f"garage rows in the band [tau-{BAND_H}, .): {int(in_band.sum()):,} "
          f"(repairable: {int((in_band & (obs_col == 0)).sum()):,})  <- rarity cell 1; gate on this n")
else:
    print("(no score column -> era audit and band gate not applicable)")

In [ ]:
# §2c — choose BAND_H / CLIP_HI on this split (policy in §1b), overriding §1's fallbacks for §3
if not (SELECT_PARAMS and HAS_SCORE):
    why = "no deciding score (v2 reality)" if not HAS_SCORE else "SELECT_PARAMS=False"
    print(f"selection skipped ({why}) -> BAND_H={BAND_H}, CLIP=[{CLIP_LO}, {CLIP_HI}]")
else:
    s_sel = corr_targets["score"].to_numpy(dtype=float)
    y_sel = obs_col.to_numpy()
    tau_sel = tau_row.to_numpy(dtype=float)      # §2b's per-row regime tau (DECIDER's LUT)

    bt = band_table(s_sel, tau_sel, y_sel, BAND_H_GRID)
    display(bt)
    figstyle.save_table(bt, f"{VERSION}_{SPLIT}_0302_sec2c_band_table")
    ok_h = bt.index[bt["cell1_pass"] & (bt.index <= H_MAX)]   # power gate under the locality cap
    if len(ok_h):
        best_h = float(ok_h[0])                  # smallest adequate width within H_MAX — see §1b
    else:
        best_h = float(BAND_H)
        wide = (" (only h > H_MAX would reach it — de-localising, §1b)"
                if bt["cell1_pass"].any() else "")
        print(f"!! no h <= {H_MAX} reaches n_cell1 >= {MIN_CELL_N}: the verified edge is thin at "
              f"every local width{wide}. Keeping BAND_H={BAND_H}; treat rarity/pnu as low-power "
              f"(§4 caveat — a label-starvation finding, not a parameter to widen).")

    cell_sel = np.where(s_sel >= tau_sel - best_h, 1 + y_sel, 3 + y_sel)
    freq_sel = pd.Series(cell_sel).value_counts(normalize=True)
    raw_mult = 1.0 / (freq_sel.size * freq_sel)  # unclipped m_c — what clip_hi guards against
    print(f"h={best_h}: unclipped m_c =",
          {int(c): round(float(v), 2) for c, v in raw_mult.sort_index().items()})

    ct = clip_table(cell_sel, CLIP_HI_GRID, CLIP_LO, raw_max=float(raw_mult.max()))
    display(ct)
    figstyle.save_table(ct, f"{VERSION}_{SPLIT}_0302_sec2c_clip_table")
    ok_hi = ct.index[ct["ess_pass"]]
    if len(ok_hi):
        best_hi = float(ok_hi.max())             # largest hi the variance budget allows — §1b
    else:
        best_hi = float(min(CLIP_HI_GRID))
        print(f"!! even the tightest clip_hi leaves ESS/n < {MIN_ESS_FRAC} — falling back to "
              f"{best_hi}; the rarity weights are variance-dominated on this split.")

    # fixed-arm context: the SAME (h, clip) applies there, keeping that contrast a pure tau one
    tau_fix = float(TAU_FIXED) if TAU_FIXED is not None else float(
        threshold.read_off(corr_targets)["tau"])
    n1_fix = int(((s_sel >= tau_fix - best_h) & (y_sel == 0)).sum())
    fix_warn = f"  !! < {MIN_CELL_N}" if n1_fix < MIN_CELL_N else ""
    print(f"fixed arm at the same h: tau={tau_fix} -> n_cell1={n1_fix}{fix_warn}")

    print(f"chosen: BAND_H {BAND_H} -> {best_h} (cap H_MAX={H_MAX}) | CLIP_LO {CLIP_LO} "
          f"(kept — can never bind, §1b) | CLIP_HI {CLIP_HI} -> {best_hi}")
    BAND_H, CLIP_HI = best_h, best_hi

## §3 — run the corrector, one file per axis

`feature_cols` comes from `config.model_features(VERSION)` — never "every column except
claim_id" (the exported matrix carries the target, and v3's its own predictions). transport/pnu
outputs hold each U claim **twice** (the (1, g) / (0, 1−g) halves); `retrain.py`'s join expands
the feature row to match, so 03_03 needs no special handling. Runs whose requirements the input
cannot meet (rarity/pnu without a score) are skipped with a printed reason, not errored.

In [ ]:
# §3 — corrector per axis -> corrected parquet + meta sidecar
FEATURE_COLS = config.model_features(VERSION)

diag_rows = []
for scheme, mode in RUNS:
    tag = run_tag(scheme, mode)
    if scheme in ("rarity", "pnu") and not HAS_SCORE:
        print(f"{tag:16s} SKIPPED — needs the deciding score, absent for {VERSION}")
        continue
    corr = ReweightCorrector(scheme=scheme, tau_mode=mode or "regime", decider=DECIDER,
                             tau=TAU_FIXED, band_h=BAND_H, clip_lo=CLIP_LO, clip_hi=CLIP_HI,
                             id_col=ID_COL)
    out, diag = corr.correct(features, corr_targets, feature_cols=FEATURE_COLS)
    p = corrected_path(tag)
    p.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(p, index=False)
    p.with_name(p.stem + "_meta.json").write_text(
        json.dumps({"version": VERSION, "split": SPLIT,
                    "features": str(FEATURES_PATH), "targets": str(TARGETS_PATH), **diag},
                   indent=2),
        encoding="utf-8")
    diag_rows.append({"run": tag, **{k: v for k, v in diag.items() if not isinstance(v, dict)}})
    print(f"{tag:16s} -> {p.name}")
diag_summary = pd.DataFrame(diag_rows).set_index("run")
display(diag_summary)
figstyle.save_table(diag_summary, f"{VERSION}_{SPLIT}_0302_sec3_diag_summary")

## §4 — what got written, and what happens next

- `src/data/real/mitigation/<v>_corrected_<split>_<tag>.parquet` — `claim_id + label + weight`
  per axis, plus a `_meta.json` sidecar (scheme, τ mode, cell table, transport diagnostics).
- **§2c's selection tables and §3's per-axis summary are also written as CSV**, named
  `<v>_<split>_0302_sec2c_band_table.csv`, `..._sec2c_clip_table.csv`, `..._sec3_diag_summary.csv`
  — via `figstyle.save_table()`, under `figures/mitigation/03_02/<v>/` (2026-09-16: moved out of
  `src/data/real/mitigation/`, which now holds only the `corrected` parquet + meta) — the values
  are already inside each `_meta.json`, but a flat CSV is faster to open directly for a quick
  look across axes without parsing JSON. `0302` names this notebook, `secN` the section,
  matching 03_01's naming convention.
- Next: **03_03_retrain.ipynb** on the version's own kernel (env-v2 / env-v3, separately) —
  it discovers these files by name and retrains one model per axis, then scores the splits.

Caveats to carry: 03_01 already dropped treatment-uncovered rows once, for every consumer alike;
the `fixed` τ arm reads one pooled boundary across eras **by design** (the
clean-single-threshold sensitivity), while `regime` is the faithful arm; if §2c could not reach
n(cell 1) ≥ `MIN_CELL_N` within `H_MAX`, the rarity scheme's cell-1 up-weighting rests on those
few rows — a label-starvation finding to report, not a reason to widen the band (§1b).
`band_h`/`clip` are §2c's choice, made on power/variance diagnostics only (never on
corrected-data metrics — §1b) and recorded in each `_meta.json`; 03_04 treats them as
sensitivity axes, not tuned constants.

## §5 — [FIG 4.6] counterfactual repairable fraction in the scrapped region

**What this checks.** Every scrapped claim's true outcome is unobserved by construction
(no garage verification once a car is scrapped, README/CLAUDE.md) -- the whole reason SFP
detection exists. This figure is the closest thing to a look inside that black box: the
transport scheme already fits `g(x) = P(observed=1 | x)` on garage rows and scores every
scrapped claim with it (Section~subsec:transport), so `1-g(x)` is the model's OWN estimate
of "how likely was this specific scrapped car actually repairable". Two questions this
answers: (1) does the scrapped region contain a non-trivial estimated-repairable mass at
all (panel A) -- if `1-g(x)` is uniformly near 0, the correction has nothing to correct, and
if it is not, that is evidence the scrap decision is cutting into repairable cars; (2) does
that estimate move the way it should (panel B) -- claims scored just above tau should look
more repairable than claims scored deep into the scrap region, since tau is where the model
itself was least confident. If panel B is flat or backwards, that is a red flag about g(x)
itself, not about the loop.

**Mechanics.** For the plain `transport` scheme every scrapped (U) claim_id appears twice in
the corrected parquet this notebook's own §3 just wrote: `(label=1, weight=g(x))` and
`(label=0, weight=1-g(x))`. The second row's weight IS the estimated counterfactual
repairable probability -- no retrain, no new model, just reading that file back.

**Panel B's x-axis, spelled out here since the plot itself only says "score decile":**
scrapped claims are sorted by score and cut into 10 equal-sized groups; decile 1 is the
lowest-scoring tenth (closest to tau, the boundary rows), decile 10 the highest-scoring tenth
(deepest into the scrap region). Only drawn when this version's `corrector_targets` carries a
score (`HAS_SCORE`). If the scrapped population's scores cluster tightly near tau, `qcut`'s
`duplicates="drop"` collapses to fewer than 10 bins -- the cell prints a note when that
happens so a sparse-looking panel B is not mistaken for a bug.

**Epistemic status.** This is a label-free, model-based estimate, not a verified rate -- read
it as "what the transport model itself implies", the same status as the corrector's own `g`
diagnostics in each `_meta.json`, never as ground truth. The image itself carries only bare
axis labels ($1-g(x)$, score decile) on purpose -- version, $n$, the mean, and everything
above go in the LaTeX caption ([FIG 4.6], `report/paper`), not baked into the PNG.

In [ ]:
# §5 — FIG 4.6: 1-g(x) over the scrapped (U) population, from the transport scheme's own output
transport_path = corrected_path("transport")
if not transport_path.is_file():
    print(f"skip FIG 4.6 -- {transport_path.name} not found (the 'transport' scheme did not "
          f"run above; it never needs a score, so this should not happen -- check §3's log)")
else:
    tr = pd.read_parquet(transport_path)
    u = (tr.merge(corr_targets[[ID_COL, "decision"]], on=ID_COL)
           .loc[lambda f: (f["decision"] == 1) & (f["label"] == 0)])   # the (0, 1-g) half of U
    repairable_frac = u["weight"].to_numpy(dtype=float)   # = 1 - g(x), one value per U claim
    print(f"{VERSION} {SPLIT}: {len(repairable_frac):,} scrapped claims | estimated "
          f"counterfactual repairable fraction: mean={repairable_frac.mean():.4f}, "
          f"median={np.median(repairable_frac):.4f}")

    # Titles/long labels stay OUT of the image on purpose -- the version, n, mean, and what
    # "decile" means belong in the LaTeX caption ([FIG 4.6], report/paper), not baked into
    # the PNG. constrained_layout replaces a manual tight_layout() call and is what fixed the
    # right panel getting clipped against the figure edge.
    fig, axes = plt.subplots(1, 2, figsize=figstyle.FIG_2, constrained_layout=True)
    axes[0].hist(repairable_frac, bins=40, color=figstyle.PRIMARY, alpha=0.85)
    axes[0].axvline(float(repairable_frac.mean()), ls="--", lw=1, color=figstyle.INK)
    axes[0].set_xlabel(r"$1-g(x)$  ($\approx\hat{P}(\mathrm{repairable}\mid x)$)")
    axes[0].set_ylabel("count")

    if HAS_SCORE:
        s_map = corr_targets.set_index(ID_COL)["score"]
        s_u = s_map.reindex(u[ID_COL]).to_numpy(dtype=float)
        decile = pd.qcut(s_u, q=10, labels=False, duplicates="drop")
        band = pd.DataFrame({"score_decile": decile, "repairable_frac": repairable_frac})
        band_summary = (band.groupby("score_decile", observed=True)["repairable_frac"]
                        .agg(["mean", "count"]).sort_index())
        if len(band_summary) < 10:
            print(f"  note: scores collapsed to {len(band_summary)} distinct bin(s), not 10 -- "
                  f"the scrapped population's scores are tightly clustered near tau (qcut's "
                  f"duplicates='drop'); panel B will look sparse, not broken.")
        display(band_summary.round(4))
        figstyle.save_table(band_summary,
                            f"{VERSION}_{SPLIT}_0302_sec5_repairable_by_score_decile")
        axes[1].plot(band_summary.index + 1, band_summary["mean"], marker="o",
                     color=figstyle.PRIMARY)
        axes[1].set_xticks(band_summary.index + 1)
        axes[1].set_xlabel("score decile")
        axes[1].set_ylabel(r"mean $1-g(x)$")
    else:
        axes[1].axis("off")
        axes[1].text(0.5, 0.5, "no deciding score for this version's\ncorrector_targets "
                     "(HAS_SCORE=False)", ha="center", va="center", fontsize=9,
                     color=figstyle.MUTED, transform=axes[1].transAxes)

    figstyle.save_fig(fig, f"{VERSION}_{SPLIT}_0302_sec5_repairable_fraction_scrapped")
    plt.show()